# Similitud estructural de elementos de la ontología

In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
import sbc_tools as sbc
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

In [28]:
graph = sbc.load("ontologia_generos_cinematograficos_completa.ttl", format="turtle")


In [29]:
sbc.print_ontology_stats(graph)


=== Estadísticas de la Ontología ===
Total de triples: 16501
Clases OWL: 718
Propiedades de objeto: 0
Instancias: 973
Relaciones de subclase: 869


In [30]:
sbc.print_class_hierarchy(graph)


JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Mexploitation (Q4044177)
  📁 Narco pelicula (Q104851179)
📁 sexploitation film (Q2292320)
  📁 Mexican sex comedy (Q5769572)
    └─ Q124051837: El Sexo Sentido
📁 Film (Q11424)
  📁 film compilation (Q101973724)
  📁 film remake (Q104840802)
  📁 carnival film (Q10543972)
  📁 4D film (Q1060398)
  📁 minimalist film (Q107210977)
  📁 maximalist film (Q107210978)
  📁 Pan-India films (Q107418888)
  📁 instant-noodles film (Q10805364)
  📁 film produced by this person (Q109108478)
  📁 film with screenplay by this person (Q109122021)
  📁 film with cinematography by this person (Q109323491)
  📁 film reboot (Q111241092)
  📁 film based on work by this person (Q111590800)
  📁 fast motion (Q111745923)
  📁 Tom and Jerry film (Q112158242)
  📁 DC Comics film (Q112318664)
  📁 Sámi-language film (Q116443876)
  📁 Shot-on-video film (Q120776065)
  📁 variety film (Q121555337)
  📁 special-venue film (Q122998965)
  📁 Lek Chalali Sasarla (Q123910751)
  📁 double feature (Q125141

In [31]:
peliculas_interes = ['Q24953', 'Q106665587', 'Q930622', 'Q135619158', 'Q121840231', 'Q426332', 'Q2364466', 'Q943961', 'Q108543', 'Q211009', 'Q18342910', 'Q20653259', 'Q15808646']

for pelicula in peliculas_interes:
    uri_entity = URIRef('http://www.wikidata.org/entity/'+pelicula)
    label =sbc._get_label(graph, uri_entity)
    print(f"Película: {label} ({pelicula})")

Película: Monty Python’s Life of Brian (Q24953)
Película: Ghostbuser: Misteri Desa Penari (Q106665587)
Película: The Moon Riders (Q930622)
Película: Goodbye Mommy (Q135619158)
Película: Chaos Theory (Q121840231)
Película: Les Bonnes Femmes (Q426332)
Película: World in My Corner (Q2364466)
Película: Blackie the Pirate (Q943961)
Película: Alien 3 (Q108543)
Película: Van Helsing (Q211009)
Película: Geostorm (Q18342910)
Película: Tour de Force (Q20653259)
Película: Let There Be Light (Q15808646)


# Implementamos las métricas de cálculo de similitud

In [32]:
from rdflib import Graph, Namespace
import math

class LocalSemanticSimilarityCalculator:
    """
    Calculadora de similitud semántica que trabaja con ontologías locales
    usando consultas SPARQL sobre el grafo RDF local.
    
    Similar a SemanticSimilarityCalculator pero ejecuta SPARQL localmente
    en lugar de consultar endpoints remotos.
    """
    
    def __init__(self, graph):
        """
        Inicializa el calculador con un grafo RDF local
        
        Args:
            graph: objeto rdflib.Graph con la ontología cargada
        """
        self.graph = graph
        self.cache = {}
        
        print(f"Calculador inicializado con {len(self.graph)} triples")
        print(f"Usando consultas SPARQL sobre grafo local\n")
    
    def execute_query(self, query):
        """Ejecuta consulta SPARQL sobre el grafo local con caché"""
        cache_key = hash(query)
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        try:
            results = self.graph.query(query)
            result_list = list(results)
            self.cache[cache_key] = result_list
            return result_list
        except Exception as e:
            print(f"Error en consulta SPARQL: {e}")
            return []
    
    def get_least_common_subsumer(self, entity1_qid, entity2_qid):
        """
        Encuentra el Least Common Subsumer (LCS) entre dos entidades
        usando consultas SPARQL sobre el grafo local
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT ?lcs ?lcsLabel WHERE {{
          # Obtener las clases de las dos entidades
          wd:{entity1_qid} rdf:type ?class1 .
          wd:{entity2_qid} rdf:type ?class2 .
          
          # Encontrar ancestros comunes
          ?class1 rdfs:subClassOf* ?lcs .
          ?class2 rdfs:subClassOf* ?lcs .
          
          # Filtrar para obtener el más específico (sin subclases más específicas comunes)
          FILTER NOT EXISTS {{
            ?sublcs rdfs:subClassOf+ ?lcs .
            ?class1 rdfs:subClassOf* ?sublcs .
            ?class2 rdfs:subClassOf* ?sublcs .
          }}
          
          OPTIONAL {{ ?lcs rdfs:label ?lcsLabel . }}
        }}
        LIMIT 1
        """
        
        results = self.execute_query(query)
        if results:
            row = results[0]
            lcs_uri = str(row.lcs)
            lcs_qid = lcs_uri.split('/')[-1]
            lcs_label = str(row.lcsLabel) if row.lcsLabel else "Unknown"
            return lcs_qid, lcs_label
        return None, None
    
    def get_depth(self, entity_qid):
        """
        Calcula la profundidad de una entidad en la jerarquía
        usando consultas SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT (COUNT(DISTINCT ?parent) as ?depth) WHERE {{
          wd:{entity_qid} rdf:type ?class .
          ?class rdfs:subClassOf* ?parent .
        }}
        """
        
        results = self.execute_query(query)
        if results:
            return int(results[0].depth)
        return 0
    
    def get_shortest_path_length(self, entity1_qid, entity2_qid):
        """
        Calcula la longitud del camino más corto entre dos entidades
        """
        lcs_qid, _ = self.get_least_common_subsumer(entity1_qid, entity2_qid)
        if not lcs_qid:
            return float('inf')
        
        depth1 = self.get_depth(entity1_qid)
        depth2 = self.get_depth(entity2_qid)
        depth_lcs = self.get_depth(lcs_qid)
        
        path_length = (depth1 - depth_lcs) + (depth2 - depth_lcs)
        return path_length
    
    def path_similarity(self, entity1_qid, entity2_qid):
        """
        Similitud Path (Rada et al., 1989)
        sim = 1 / (1 + shortest_path)
        """
        path_length = self.get_shortest_path_length(entity1_qid, entity2_qid)
        if path_length == float('inf'):
            return 0.0
        return 1.0 / (1.0 + path_length)
    
    def wu_palmer_similarity(self, entity1_qid, entity2_qid):
        """
        Similitud Wu & Palmer (1994)
        sim = 2 * depth(lcs) / (depth(c1) + depth(c2))
        """
        lcs_qid, _ = self.get_least_common_subsumer(entity1_qid, entity2_qid)
        if not lcs_qid:
            return 0.0
        
        depth1 = self.get_depth(entity1_qid)
        depth2 = self.get_depth(entity2_qid)
        depth_lcs = self.get_depth(lcs_qid)
        
        if depth1 + depth2 == 0:
            return 0.0
        
        return (2.0 * depth_lcs) / (depth1 + depth2)
    
    def get_properties(self, entity_qid):
        """
        Obtiene todas las propiedades de una entidad usando SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        
        SELECT DISTINCT ?property WHERE {{
          wd:{entity_qid} ?property ?value .
          FILTER(STRSTARTS(STR(?property), STR(wdt:)))
        }}
        """
        
        results = self.execute_query(query)
        return set(str(row.property) for row in results)
    
    def get_property_values(self, entity_qid):
        """
        Obtiene pares (propiedad, valor) de una entidad usando SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        
        SELECT ?property ?value WHERE {{
          wd:{entity_qid} ?property ?value .
          FILTER(STRSTARTS(STR(?property), STR(wdt:)))
        }}
        """
        
        results = self.execute_query(query)
        return [(str(row.property), str(row.value)) for row in results]
    
    def jaccard_property_similarity(self, entity1_qid, entity2_qid):
        """
        Similitud Jaccard basada en propiedades
        sim = |props(e1) ∩ props(e2)| / |props(e1) ∪ props(e2)|
        """
        props1 = self.get_properties(entity1_qid)
        props2 = self.get_properties(entity2_qid)
        
        if not props1 and not props2:
            return 0.0
        
        intersection = len(props1 & props2)
        union = len(props1 | props2)
        
        if union == 0:
            return 0.0
        
        return intersection / union
    
    def shared_value_similarity(self, entity1_qid, entity2_qid):
        """
        Similitud basada en valores compartidos usando SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        
        SELECT (COUNT(DISTINCT ?property) as ?sharedProps)
               (COUNT(DISTINCT ?sharedValue) as ?sharedValues)
        WHERE {{
          {{
            SELECT DISTINCT ?property ?sharedValue WHERE {{
              wd:{entity1_qid} ?property ?sharedValue .
              wd:{entity2_qid} ?property ?sharedValue .
              FILTER(STRSTARTS(STR(?property), STR(wdt:)))
            }}
          }}
        }}
        """
        
        results = self.execute_query(query)
        if not results:
            return 0.0
        
        # Obtener total de valores únicos
        pairs1 = set(self.get_property_values(entity1_qid))
        pairs2 = set(self.get_property_values(entity2_qid))
        
        if not pairs1 and not pairs2:
            return 0.0
        
        shared = len(pairs1 & pairs2)
        total = len(pairs1 | pairs2)
        
        if total == 0:
            return 0.0
        
        return shared / total
    
    
    def get_all_entities(self):
        """Obtiene todas las entidades (instancias) del grafo usando SPARQL"""
        query = """
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        PREFIX wd: <http://www.wikidata.org/entity/>
        
        SELECT DISTINCT ?entity WHERE {
          ?entity rdf:type ?class .
          
          # Filtrar clases y propiedades
          FILTER NOT EXISTS { ?entity rdf:type owl:Class }
          FILTER NOT EXISTS { ?entity rdf:type owl:ObjectProperty }
          
          # Solo entidades de Wikidata
          FILTER(STRSTARTS(STR(?entity), STR(wd:)))
        }
        """
        
        results = self.execute_query(query)
        entities = []
        for row in results:
            entity_uri = str(row.entity)
            qid = entity_uri.split('/')[-1]
            if qid.startswith('Q') and qid[1:].isdigit():
                entities.append(qid)
        
        return entities
    
    def get_label(self, entity_qid):
        """Obtiene el label de una entidad usando SPARQL"""
        query = f"""
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        
        SELECT ?label WHERE {{
          wd:{entity_qid} rdfs:label ?label .
        }}
        LIMIT 1
        """
        
        results = self.execute_query(query)
        if results:
            return str(results[0].label)
        return entity_qid
    
    def find_similar_entities(self, target_qid, top_k=5):
        """
        Encuentra las K entidades más similares a la entidad objetivo
        """
        all_entities = self.get_all_entities()
        candidates = [e for e in all_entities if e != target_qid]
        
        similarities = []
        for candidate_qid in candidates:
            try:
                sim = self.hybrid_similarity(target_qid, candidate_qid)
                label = self.get_label(candidate_qid)
                
                similarities.append({
                    'qid': candidate_qid,
                    'label': label,
                    'similarity': sim['total'],
                    'details': sim
                })
            except Exception as e:
                continue
        
        similarities.sort(key=lambda x: x['similarity'], reverse=True)
        return similarities[:top_k]


In [33]:
# Inicializar calculadora de similitud semántica local con el grafo generado
semantic_sim = LocalSemanticSimilarityCalculator(graph)


Calculador inicializado con 16501 triples
Usando consultas SPARQL sobre grafo local



## Hacemos unas cuentas pruebas

### Profundidad de las películas de interés


In [34]:
for pelicula in peliculas_interes:
    uri_entity = URIRef('http://www.wikidata.org/entity/'+pelicula)
    label =sbc._get_label(graph, uri_entity)
    depth = semantic_sim.get_depth(pelicula)  
    print(f"Película: {label} ({pelicula}), Profundidad: {depth}")

Película: Monty Python’s Life of Brian (Q24953), Profundidad: 3
Película: Ghostbuser: Misteri Desa Penari (Q106665587), Profundidad: 3
Película: The Moon Riders (Q930622), Profundidad: 3
Película: Goodbye Mommy (Q135619158), Profundidad: 4
Película: Chaos Theory (Q121840231), Profundidad: 3
Película: Les Bonnes Femmes (Q426332), Profundidad: 3
Película: World in My Corner (Q2364466), Profundidad: 3
Película: Blackie the Pirate (Q943961), Profundidad: 3
Película: Alien 3 (Q108543), Profundidad: 4
Película: Van Helsing (Q211009), Profundidad: 4
Película: Geostorm (Q18342910), Profundidad: 6
Película: Tour de Force (Q20653259), Profundidad: 2
Película: Let There Be Light (Q15808646), Profundidad: 5


### LCS entre algunas películas de interés


In [35]:
semantic_sim.get_least_common_subsumer("Q24953", "Q106665587")  # LCS entre "Monty Python’s Life of Brian" y "Ghostbuser: Misteri Desa Penari" 

('Q157443', 'comedy film')

In [36]:
semantic_sim.get_least_common_subsumer("Q24953", "Q211009")  # LCS entre "Monty Python’s Life of Brian" y "Van Helsing" 

('Q11424', 'Film')

Probamos todas con todas

In [37]:
# Initialize a 2D matrix with "Film" as default values
lcs_matrix = [['Film' for _ in peliculas_interes] for _ in peliculas_interes]
# Populate the matrix with the similarity values
for i, p1 in enumerate(peliculas_interes):
    for j, p2 in enumerate(peliculas_interes):
        lcs = semantic_sim.get_least_common_subsumer(p1, p2)[1]
        lcs_matrix[i][j] = lcs


### Shotest Path Similarity

In [38]:
semantic_sim.get_shortest_path_length("Q24953", "Q106665587")  # Longitud del camino más corto


4

Probamos todas con todas. Aprovechamos para crear una función que nos ayude

In [39]:
def compute_matrix(function, labels):
    """Compute a similarity matrix using the provided function"""
    # Initialize a 2D matrix with zeros
    matrix = [[0 for _ in labels] for _ in labels]
    # Populate the matrix with the similarity values
    for i, p1 in enumerate(labels):
        for j, p2 in enumerate(labels):
            value = function(p1, p2)
            matrix[i][j] = f'{value:.2f}'
    return matrix

In [40]:
shortest_path_matrix = compute_matrix(semantic_sim.get_shortest_path_length, peliculas_interes)
sbc.print_matrix(peliculas_interes, shortest_path_matrix, 15)

               Q24953         Q106665587     Q930622        Q135619158     Q121840231     Q426332        Q2364466       Q943961        Q108543        Q211009        Q18342910      Q20653259      Q15808646      
Q24953         4.00            4.00            4.00            5.00            4.00            4.00            4.00            4.00            5.00            5.00            7.00            3.00            6.00            
Q106665587     4.00            4.00            4.00            5.00            4.00            4.00            4.00            4.00            5.00            5.00            7.00            3.00            6.00            
Q930622        4.00            4.00            4.00            5.00            4.00            4.00            4.00            4.00            5.00            5.00            7.00            3.00            6.00            
Q135619158     5.00            5.00            5.00            6.00            5.00            5.00            5.00  

### Path similarity

In [41]:
semantic_sim.path_similarity("Q24953", "Q106665587")  # Similitud Path

0.2

In [42]:
path_similarity_matrix = compute_matrix(semantic_sim.path_similarity, peliculas_interes)
sbc.print_matrix(peliculas_interes, path_similarity_matrix, 15)

               Q24953         Q106665587     Q930622        Q135619158     Q121840231     Q426332        Q2364466       Q943961        Q108543        Q211009        Q18342910      Q20653259      Q15808646      
Q24953         0.20            0.20            0.20            0.17            0.20            0.20            0.20            0.20            0.17            0.17            0.12            0.25            0.14            
Q106665587     0.20            0.20            0.20            0.17            0.20            0.20            0.20            0.20            0.17            0.17            0.12            0.25            0.14            
Q930622        0.20            0.20            0.20            0.17            0.20            0.20            0.20            0.20            0.17            0.17            0.12            0.25            0.14            
Q135619158     0.17            0.17            0.17            0.14            0.17            0.17            0.17  

### WU & PALMER similarity

In [43]:
semantic_sim.wu_palmer_similarity("Q24953", "Q106665587")  # Similitud Wu & Palmer

0.3333333333333333

In [44]:
wu_palmer_similarity_matrix = compute_matrix(semantic_sim.wu_palmer_similarity, peliculas_interes)
sbc.print_matrix(peliculas_interes, wu_palmer_similarity_matrix, 15)

               Q24953         Q106665587     Q930622        Q135619158     Q121840231     Q426332        Q2364466       Q943961        Q108543        Q211009        Q18342910      Q20653259      Q15808646      
Q24953         0.33            0.33            0.33            0.29            0.33            0.33            0.33            0.33            0.29            0.29            0.22            0.40            0.25            
Q106665587     0.33            0.33            0.33            0.29            0.33            0.33            0.33            0.33            0.29            0.29            0.22            0.40            0.25            
Q930622        0.33            0.33            0.33            0.29            0.33            0.33            0.33            0.33            0.29            0.29            0.22            0.40            0.25            
Q135619158     0.29            0.29            0.29            0.25            0.29            0.29            0.29  

### Jaccard similarity

In [68]:
semantic_sim.jaccard_property_similarity("Q24953", "Q106665587")  # Similitud Jaccard basada en propiedades

1.0

In [46]:
jaccard_property_similarity_matrix = compute_matrix(semantic_sim.jaccard_property_similarity, peliculas_interes)
sbc.print_matrix(peliculas_interes, jaccard_property_similarity_matrix, 15)

               Q24953         Q106665587     Q930622        Q135619158     Q121840231     Q426332        Q2364466       Q943961        Q108543        Q211009        Q18342910      Q20653259      Q15808646      
Q24953         1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            
Q106665587     1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            
Q930622        1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            1.00            
Q135619158     1.00            1.00            1.00            1.00            1.00            1.00            1.00  

---
# Ejercicio

## Aplica las métricas de similitud estructural a tu grafo de conocimiento.

## Analiza los resultados y aplicabilidad

In [71]:
from rdflib import Graph, Namespace
import math
import sbc_tools as sbc
class LocalSemanticSimilarityCalculator_Propio:
    """
    Calculadora de similitud semántica que trabaja con ontologías locales
    usando consultas SPARQL sobre el grafo RDF local.
    
    Similar a SemanticSimilarityCalculator pero ejecuta SPARQL localmente
    en lugar de consultar endpoints remotos.
    """
    
    def __init__(self, graph):
        """
        Inicializa el calculador con un grafo RDF local
        
        Args:
            graph: objeto rdflib.Graph con la ontología cargada
        """
        self.graph = graph
        self.cache = {}
        
        print(f"Calculador inicializado con {len(self.graph)} triples")
        print(f"Usando consultas SPARQL sobre grafo local\n")
    
    def execute_query(self, query):
        """Ejecuta consulta SPARQL sobre el grafo local con caché"""
        cache_key = hash(query)
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        try:
            results = self.graph.query(query)
            result_list = list(results)
            self.cache[cache_key] = result_list
            return result_list
        except Exception as e:
            print(f"Error en consulta SPARQL: {e}")
            return []
    
    def get_least_common_subsumer(self, entity1_qid, entity2_qid, tag_p='onto'):
        """
        Encuentra el Least Common Subsumer (LCS) entre dos entidades
        usando consultas SPARQL sobre el grafo local
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX onto: <http://www.fdi.ucm.es/mia/sbc/animeontology/>
        PREFIX ns2: <http://www.semanticweb.org/eladi/ontologies/anime-ontology#>

        SELECT ?lcs ?lcsLabel WHERE {{
          # Obtener las clases de las dos entidades
          {tag_p}:{entity1_qid} rdf:type ?class1 .
          {tag_p}:{entity2_qid} rdf:type ?class2 .
          
          # Encontrar ancestros comunes
          ?class1 rdfs:subClassOf* ?lcs .
          ?class2 rdfs:subClassOf* ?lcs .
          
          # Filtrar para obtener el más específico (sin subclases más específicas comunes)
          FILTER NOT EXISTS {{
            ?sublcs rdfs:subClassOf+ ?lcs .
            ?class1 rdfs:subClassOf* ?sublcs .
            ?class2 rdfs:subClassOf* ?sublcs .
          }}
          
          OPTIONAL {{ ?lcs rdfs:label ?lcsLabel . }}
        }}
        LIMIT 1
        """
        
        results = self.execute_query(query)
        if results:
            row = results[0]
            lcs_uri = str(row.lcs)
            lcs_qid = lcs_uri.split('/')[-1]
            lcs_label = str(row.lcsLabel) if row.lcsLabel else "Unknown"
            return lcs_qid, lcs_label
        return None, None
    
    def get_depth(self, entity_qid, tag_p='onto'):
        """
        Calcula la profundidad de una entidad en la jerarquía
        usando consultas SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX onto: <http://www.fdi.ucm.es/mia/sbc/animeontology/>
        PREFIX ns2: <http://www.semanticweb.org/eladi/ontologies/anime-ontology#>

        SELECT (COUNT(DISTINCT ?parent) as ?depth) WHERE {{
          {tag_p}:{entity_qid} rdf:type ?class .
          ?class rdfs:subClassOf* ?parent .
        }}
        """
        
        results = self.execute_query(query)
        if results:
            return int(results[0].depth)
        return 0
    
    def get_shortest_path_length(self, entity1_qid, entity2_qid):
        """
        Calcula la longitud del camino más corto entre dos entidades
        """
        lcs_qid, _ = self.get_least_common_subsumer(entity1_qid, entity2_qid)
        if not lcs_qid:
            return float('inf')
        
        depth1 = self.get_depth(entity1_qid)
        depth2 = self.get_depth(entity2_qid)
        depth_lcs = self.get_depth(lcs_qid)
        
        path_length = (depth1 - depth_lcs) + (depth2 - depth_lcs)
        return path_length
    
    def path_similarity(self, entity1_qid, entity2_qid):
        """
        Similitud Path (Rada et al., 1989)
        sim = 1 / (1 + shortest_path)
        """
        path_length = self.get_shortest_path_length(entity1_qid, entity2_qid)
        if path_length == float('inf'):
            return 0.0
        return 1.0 / (1.0 + path_length)
    
    def wu_palmer_similarity(self, entity1_qid, entity2_qid):
        """
        Similitud Wu & Palmer (1994)
        sim = 2 * depth(lcs) / (depth(c1) + depth(c2))
        """
        lcs_qid, _ = self.get_least_common_subsumer(entity1_qid, entity2_qid)
        if not lcs_qid:
            return 0.0
        
        depth1 = self.get_depth(entity1_qid)
        depth2 = self.get_depth(entity2_qid)
        depth_lcs = self.get_depth(lcs_qid)
        
        if depth1 + depth2 == 0:
            return 0.0
        
        return (2.0 * depth_lcs) / (depth1 + depth2)
    
    def get_properties(self, entity_qid):
        """
        Obtiene todas las propiedades de una entidad usando SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX onto: <http://www.fdi.ucm.es/mia/sbc/animeontology/>
        PREFIX ns2: <http://www.semanticweb.org/eladi/ontologies/anime-ontology#>

        SELECT DISTINCT ?property WHERE {{
          onto:{entity_qid} ?property ?value .
        }}
        """
        
        results = self.execute_query(query)
        return set(str(row.property) for row in results)
    
    def get_property_values(self, entity_qid, tag_p='onto'):
        """
        Obtiene pares (propiedad, valor) de una entidad usando SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX onto: <http://www.fdi.ucm.es/mia/sbc/animeontology/>
        PREFIX ns2: <http://www.semanticweb.org/eladi/ontologies/anime-ontology#>

        SELECT ?property ?value WHERE {{
          {tag_p}:{entity_qid} ?property ?value .
          FILTER(STRSTARTS(STR(?property), STR(wdt:)))
        }}
        """
        
        results = self.execute_query(query)
        return [(str(row.property), str(row.value)) for row in results]
    
    def jaccard_property_similarity(self, entity1_qid, entity2_qid):
        """
        Similitud Jaccard basada en propiedades
        sim = |props(e1) ∩ props(e2)| / |props(e1) ∪ props(e2)|
        """
        props1 = self.get_properties(entity1_qid)
        props2 = self.get_properties(entity2_qid)
        
        if not props1 and not props2:
            return 0.0
        
        intersection = len(props1 & props2)
        union = len(props1 | props2)
        
        if union == 0:
            return 0.0
        
        return intersection / union
    
    def shared_value_similarity(self, entity1_qid, entity2_qid, tag_p='onto'):
        """
        Similitud basada en valores compartidos usando SPARQL
        """
        query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX onto: <http://www.fdi.ucm.es/mia/sbc/animeontology/>
        PREFIX ns2: <http://www.semanticweb.org/eladi/ontologies/anime-ontology#>

        SELECT (COUNT(DISTINCT ?property) as ?sharedProps)
               (COUNT(DISTINCT ?sharedValue) as ?sharedValues)
        WHERE {{
          {{
            SELECT DISTINCT ?property ?sharedValue WHERE {{
              {tag_p}:{entity1_qid} ?property ?sharedValue .
              {tag_p}:{entity2_qid} ?property ?sharedValue .
              FILTER(STRSTARTS(STR(?property), STR(wdt:)))
            }}
          }}
        }}
        """
        
        results = self.execute_query(query)
        if not results:
            return 0.0
        
        # Obtener total de valores únicos
        pairs1 = set(self.get_property_values(entity1_qid))
        pairs2 = set(self.get_property_values(entity2_qid))
        
        if not pairs1 and not pairs2:
            return 0.0
        
        shared = len(pairs1 & pairs2)
        total = len(pairs1 | pairs2)
        
        if total == 0:
            return 0.0
        
        return shared / total
    
    
    def get_all_entities(self):
        """Obtiene todas las entidades (instancias) del grafo usando SPARQL"""
        query = """
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX onto: <http://www.fdi.ucm.es/mia/sbc/animeontology/>
        PREFIX ns2: <http://www.semanticweb.org/eladi/ontologies/anime-ontology#>

        SELECT DISTINCT ?entity WHERE {
          ?entity rdf:type ?class .
          
          # Filtrar clases y propiedades
          FILTER NOT EXISTS { ?entity rdf:type owl:Class }
          FILTER NOT EXISTS { ?entity rdf:type owl:ObjectProperty }
          
          # Solo entidades de Wikidata
          FILTER(STRSTARTS(STR(?entity), STR(wd:)))
        }
        """
        
        results = self.execute_query(query)
        entities = []
        for row in results:
            entity_uri = str(row.entity)
            qid = entity_uri.split('/')[-1]
            if qid.startswith('Q') and qid[1:].isdigit():
                entities.append(qid)
        
        return entities
    
    def get_label(self, entity_qid, tag_p='onto'):
        """Obtiene el label de una entidad usando SPARQL"""
        query = f"""
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX onto: <http://www.fdi.ucm.es/mia/sbc/animeontology/>
        PREFIX ns2: <http://www.semanticweb.org/eladi/ontologies/anime-ontology#>

        SELECT ?label WHERE {{
          {tag_p}:{entity_qid} rdfs:label ?label .
        }}
        LIMIT 1
        """
        
        results = self.execute_query(query)
        if results:
            return str(results[0].label)
        return entity_qid
    
    def find_similar_entities(self, target_qid, top_k=5):
        """
        Encuentra las K entidades más similares a la entidad objetivo
        """
        all_entities = self.get_all_entities()
        candidates = [e for e in all_entities if e != target_qid]
        
        similarities = []
        for candidate_qid in candidates:
            try:
                sim = self.hybrid_similarity(target_qid, candidate_qid)
                label = self.get_label(candidate_qid)
                
                similarities.append({
                    'qid': candidate_qid,
                    'label': label,
                    'similarity': sim['total'],
                    'details': sim
                })
            except Exception as e:
                continue
        
        similarities.sort(key=lambda x: x['similarity'], reverse=True)
        return similarities[:top_k]


In [72]:
# cargamos nuestro grafo
graph_propio = sbc.load("anime_knowledge_graph_clean.ttl", format="turtle" )

animes_interes = ['Cowboy_Bebop', 'Rurouni_Kenshin', 'Oh_My_Goddess', 'Angel_Sanctuary', 'Zipang', 'Neon_Genesis_Evangelion_The_End_of_Evangelion', 'Initial_D_Fourth_Stage']

semantic_sim_propio = LocalSemanticSimilarityCalculator_Propio(graph_propio)

Calculador inicializado con 50386 triples
Usando consultas SPARQL sobre grafo local



In [73]:
# creamos las similitudes
semantic_sim_propio.get_least_common_subsumer("Cowboy_Bebop", "Rurouni_Kenshin")

('action_anime', 'Action (Genre)')

In [74]:
lcs_matrix = [['Anime Serie' for _ in animes_interes] for _ in animes_interes]
for i, p1 in enumerate(animes_interes):
    for j, p2 in enumerate(animes_interes):
        lcs = semantic_sim_propio.get_least_common_subsumer(p1, p2)[1]
        lcs_matrix[i][j] = lcs

sbc.print_matrix(animes_interes, lcs_matrix)

                              Cowboy_Bebop                  Rurouni_Kenshin               Oh_My_Goddess                 Angel_Sanctuary               Zipang                        Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage        
Cowboy_Bebop                  Action (Genre)                 Action (Genre)                 Anime Series                   Action (Genre)                 Action (Genre)                 Anime Series                   Action (Genre)                 
Rurouni_Kenshin               Action (Genre)                 Action (Genre)                 Anime Series                   Action (Genre)                 Action (Genre)                 Anime Series                   Action (Genre)                 
Oh_My_Goddess                 Anime Series                   Anime Series                   Comedy (Genre)                 Anime Series                   Anime Series                   Anime Series                   Anime Series                   


In [75]:
semantic_sim_propio.get_shortest_path_length("Cowboy_Bebop", "Rurouni_Kenshin")  # Longitud del camino más corto

def compute_matrix(function, labels):
    """Compute a similarity matrix using the provided function"""
    # Initialize a 2D matrix with zeros
    matrix = [[0 for _ in labels] for _ in labels]
    # Populate the matrix with the similarity values
    for i, p1 in enumerate(labels):
        for j, p2 in enumerate(labels):
            value = function(p1, p2)
            matrix[i][j] = f'{value:.2f}'
    return matrix

shortest_path_matrix = compute_matrix(semantic_sim_propio.get_shortest_path_length, animes_interes)
sbc.print_matrix(animes_interes, shortest_path_matrix, 15)

               Cowboy_Bebop   Rurouni_KenshinOh_My_Goddess  Angel_SanctuaryZipang         Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage
Cowboy_Bebop   6.00            8.00            10.00           10.00           7.00            9.00            6.00            
Rurouni_Kenshin8.00            10.00           12.00           12.00           9.00            11.00           8.00            
Oh_My_Goddess  10.00           12.00           12.00           14.00           11.00           11.00           10.00           
Angel_Sanctuary10.00           12.00           14.00           14.00           11.00           13.00           10.00           
Zipang         7.00            9.00            11.00           11.00           8.00            10.00           7.00            
Neon_Genesis_Evangelion_The_End_of_Evangelion9.00            11.00           11.00           13.00           10.00           12.00           9.00            
Initial_D_Fourth_Stage6.00            8.00  

In [76]:
semantic_sim_propio.path_similarity("Cowboy_Bebop", "Angel_Sanctuary")  # Similitud Path

path_similarity_matrix = compute_matrix(semantic_sim_propio.path_similarity, animes_interes)
sbc.print_matrix(animes_interes, path_similarity_matrix, 15)

               Cowboy_Bebop   Rurouni_KenshinOh_My_Goddess  Angel_SanctuaryZipang         Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage
Cowboy_Bebop   0.14            0.11            0.09            0.09            0.12            0.10            0.14            
Rurouni_Kenshin0.11            0.09            0.08            0.08            0.10            0.08            0.11            
Oh_My_Goddess  0.09            0.08            0.08            0.07            0.08            0.08            0.09            
Angel_Sanctuary0.09            0.08            0.07            0.07            0.08            0.07            0.09            
Zipang         0.12            0.10            0.08            0.08            0.11            0.09            0.12            
Neon_Genesis_Evangelion_The_End_of_Evangelion0.10            0.08            0.08            0.07            0.09            0.08            0.10            
Initial_D_Fourth_Stage0.14            0.11  

In [77]:
semantic_sim_propio.wu_palmer_similarity("Cowboy_Bebop", "Angel_Sanctuary")  # Similitud Wu & Palmer

wu_palmer_similarity_matrix = compute_matrix(semantic_sim_propio.wu_palmer_similarity, animes_interes)
sbc.print_matrix(animes_interes, wu_palmer_similarity_matrix, 15)

               Cowboy_Bebop   Rurouni_KenshinOh_My_Goddess  Angel_SanctuaryZipang         Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage
Cowboy_Bebop   0.40            0.33            0.17            0.29            0.36            0.18            0.40            
Rurouni_Kenshin0.33            0.29            0.14            0.25            0.31            0.15            0.33            
Oh_My_Goddess  0.17            0.14            0.14            0.12            0.15            0.15            0.17            
Angel_Sanctuary0.29            0.25            0.12            0.22            0.27            0.13            0.29            
Zipang         0.36            0.31            0.15            0.27            0.33            0.17            0.36            
Neon_Genesis_Evangelion_The_End_of_Evangelion0.18            0.15            0.15            0.13            0.17            0.00            0.18            
Initial_D_Fourth_Stage0.40            0.33  

In [78]:
semantic_sim_propio.jaccard_property_similarity("Cowboy_Bebop", "Angel_Sanctuary")  # Similitud Jaccard basada en propiedades

jaccard_property_similarity_matrix = compute_matrix(semantic_sim_propio.jaccard_property_similarity, animes_interes)
sbc.print_matrix(animes_interes, jaccard_property_similarity_matrix, 15)

               Cowboy_Bebop   Rurouni_KenshinOh_My_Goddess  Angel_SanctuaryZipang         Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage
Cowboy_Bebop   1.00            1.00            1.00            1.00            0.80            1.00            0.80            
Rurouni_Kenshin1.00            1.00            1.00            1.00            0.80            1.00            0.80            
Oh_My_Goddess  1.00            1.00            1.00            1.00            0.80            1.00            0.80            
Angel_Sanctuary1.00            1.00            1.00            1.00            0.80            1.00            0.80            
Zipang         0.80            0.80            0.80            0.80            1.00            0.80            1.00            
Neon_Genesis_Evangelion_The_End_of_Evangelion1.00            1.00            1.00            1.00            0.80            1.00            0.80            
Initial_D_Fourth_Stage0.80            0.80  